# Week 1 — Python for Machine Learning & Data Preprocessing
**YuvaIntern | AI Pioneers Internship**

## Objective
Load the Titanic passenger dataset, inspect and clean it, handle missing values, encode categorical variables, select relevant features, scale numerical features, and perform exploratory data analysis (EDA).

**Beginner note:** Run the notebook from top to bottom in Google Colab. The final cells print real results from the dataset and save the cleaned data, charts, and a report-results file.

## 1. Import libraries
- **Pandas**: work with tables.
- **NumPy**: numerical operations.
- **Matplotlib / Seaborn**: charts.
- **scikit-learn**: encode categories and scale numerical values.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

print("Libraries imported successfully.")

## 2. Load the dataset
This notebook uses the public Titanic training CSV hosted on GitHub, based on the well-known Kaggle Titanic dataset. If the download fails, download `train.csv` from [Kaggle Titanic Data](https://www.kaggle.com/competitions/titanic/data), upload it to Colab, and change `DATA_URL` to `"train.csv"`.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/rebeccabilbro/titanic/master/data/train.csv"

try:
    df = pd.read_csv(DATA_URL)
except Exception as e:
    print("Automatic download failed:", e)
    from google.colab import files
    uploaded = files.upload()  # Upload train.csv from your computer
    df = pd.read_csv("train.csv")

df.to_csv("data/titanic_original.csv", index=False)

print("Original dataset shape (rows, columns):", df.shape)
display(df.head())

## 3. Inspect the data
We inspect data types, summary statistics, duplicates, and missing values before making changes. This gives us evidence for the preprocessing decisions.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null counts:")
df.info()

print("\nNumerical summary:")
display(df.describe())

duplicate_count = int(df.duplicated().sum())
missing_before = df.isna().sum().sort_values(ascending=False)
print("\nDuplicate rows:", duplicate_count)
print("\nMissing values before cleaning:")
display(missing_before[missing_before > 0].to_frame("missing_count"))

print("\nSurvival counts (0 = did not survive, 1 = survived):")
display(df["Survived"].value_counts().sort_index().rename(index={0:"Did not survive", 1:"Survived"}))

## 4. Clean and prepare the data
Decisions:
1. Remove duplicate rows, if any.
2. Fill missing `Age` values with the median because it is less affected by extreme values than the mean.
3. Fill missing `Embarked` values with the most frequent category (mode).
4. Drop `Cabin` because it has many missing values in this dataset.
5. Drop `PassengerId`, `Name`, and `Ticket` from the modelling features because they are identifiers/free-text fields and are not directly used in this beginner workflow. The original CSV is preserved.
6. Keep `Survived` as the target label for a possible later classification task.

In [ ]:
clean = df.copy()
rows_before = len(clean)
clean = clean.drop_duplicates().copy()
duplicates_removed = rows_before - len(clean)

# Handle missing values
age_median = clean["Age"].median()
clean["Age"] = clean["Age"].fillna(age_median)

embarked_mode = clean["Embarked"].mode(dropna=True)[0]
clean["Embarked"] = clean["Embarked"].fillna(embarked_mode)

# Drop columns that are not used as model features in this beginner project
drop_columns = [c for c in ["PassengerId", "Name", "Ticket", "Cabin"] if c in clean.columns]
clean = clean.drop(columns=drop_columns)

print("Duplicate rows removed:", duplicates_removed)
print("Age median used:", round(float(age_median), 3))
print("Embarked mode used:", embarked_mode)
print("Columns dropped:", drop_columns)
print("Shape after cleaning and dropping columns:", clean.shape)
print("\nRemaining missing values:")
display(clean.isna().sum().to_frame("missing_count"))

## 5. Feature selection and categorical encoding
For this demonstration, the target is `Survived`. We keep passenger class, sex, age, family counts, fare, and embarkation port as candidate features. One-hot encoding converts categories into numeric columns without pretending that categories have an ordinal ranking.

In [ ]:
target = "Survived"
candidate_features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
features = clean[candidate_features].copy()
y = clean[target].copy()

X_encoded = pd.get_dummies(
    features,
    columns=["Sex", "Embarked"],
    drop_first=False,
    dtype=int
)

print("Selected original features:", candidate_features)
print("Encoded feature columns:")
print(X_encoded.columns.tolist())
display(X_encoded.head())

## 6. Scale numerical features
StandardScaler transforms each selected numerical feature to approximately mean 0 and standard deviation 1. For a real predictive model, fit the scaler only on training data and apply it to test data to avoid data leakage. Here, we demonstrate the transformation on the cleaned dataset for learning purposes.

In [ ]:
numeric_to_scale = ["Age", "Fare"]
scaler = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[numeric_to_scale] = scaler.fit_transform(X_encoded[numeric_to_scale])

print("Numerical columns scaled:", numeric_to_scale)
display(X_scaled.head())

## 7. Exploratory Data Analysis (EDA)
The charts help us understand the dataset. They show patterns and associations, not proof that one feature causes survival.

In [ ]:
sns.set_theme()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(data=df, x="Survived", ax=axes[0])
axes[0].set_title("Passenger survival counts")
axes[0].set_xticks([0, 1], ["Did not survive", "Survived"])

sns.histplot(data=df, x="Age", bins=25, kde=True, ax=axes[1])
axes[1].set_title("Age distribution")

sns.boxplot(data=df, x="Survived", y="Fare", ax=axes[2])
axes[2].set_title("Fare by survival status")
axes[2].set_xticks([0, 1], ["Did not survive", "Survived"])

plt.tight_layout()
plt.savefig("outputs/eda_charts.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# Compare survival rate by sex and passenger class
survival_by_sex = df.groupby("Sex", dropna=False)["Survived"].mean().sort_values(ascending=False)
survival_by_class = df.groupby("Pclass")["Survived"].mean().sort_index()

print("Survival rate by sex:")
display((survival_by_sex * 100).round(2).rename("survival_rate_percent").to_frame())

print("Survival rate by passenger class:")
display((survival_by_class * 100).round(2).rename("survival_rate_percent").to_frame())

## 8. Save cleaned and encoded datasets
The cleaned CSV contains selected columns with missing values handled. The encoded CSV contains numeric features and the target. The scaled CSV contains the encoded features with `Age` and `Fare` standardized.

In [ ]:
clean.to_csv("data/titanic_cleaned.csv", index=False)

encoded_output = X_encoded.copy()
encoded_output[target] = y.values
encoded_output.to_csv("data/titanic_encoded.csv", index=False)

scaled_output = X_scaled.copy()
scaled_output[target] = y.values
scaled_output.to_csv("data/titanic_scaled.csv", index=False)

print("Saved:")
print("- data/titanic_original.csv")
print("- data/titanic_cleaned.csv")
print("- data/titanic_encoded.csv")
print("- data/titanic_scaled.csv")
print("- outputs/eda_charts.png")

## 9. Automatically create the real-results summary
Run this cell after all previous cells. It writes `outputs/results_summary.txt` with values calculated from your actual run. Copy these values into your report before submitting.

In [ ]:
survivors = int((df["Survived"] == 1).sum())
not_survivors = int((df["Survived"] == 0).sum())
survival_rate = float(df["Survived"].mean() * 100)
missing_total_before = int(df.isna().sum().sum())
missing_total_after = int(clean.isna().sum().sum())

summary = f'''
WEEK 1 PREPROCESSING RESULTS
============================
Original dataset shape: {df.shape[0]} rows x {df.shape[1]} columns
Duplicate rows found in original data: {duplicate_count}
Duplicate rows removed: {duplicates_removed}
Missing cells before preprocessing: {missing_total_before}
Missing cells after cleaning: {missing_total_after}
Age median used for imputation: {age_median:.3f}
Most frequent Embarked category used: {embarked_mode}
Columns removed from cleaned dataset: {", ".join(drop_columns)}
Cleaned dataset shape: {clean.shape[0]} rows x {clean.shape[1]} columns
Encoded feature matrix shape: {X_encoded.shape[0]} rows x {X_encoded.shape[1]} columns
Scaled feature matrix shape: {X_scaled.shape[0]} rows x {X_scaled.shape[1]} columns
Passengers who survived: {survivors}
Passengers who did not survive: {not_survivors}
Overall survival rate in this dataset: {survival_rate:.2f}%

Interpretation notes:
- Survival counts show the target class distribution in this sample.
- Age and Fare were standardized for demonstration.
- One-hot encoding was used for Sex and Embarked.
- The survival rate describes this dataset; it is not a causal conclusion.
'''
Path("outputs/results_summary.txt").write_text(summary, encoding="utf-8")
print(summary)

## 10. Download your outputs from Colab
Use the left **Files** panel in Colab to download the `data` and `outputs` folders/files. You can also create a ZIP of all outputs with this final cell.

In [ ]:
import shutil
shutil.make_archive("week1_outputs", "zip", ".")
print("Created week1_outputs.zip. Download it from the Colab Files panel.")